In [1]:
import pandas as pd
import numpy as np

traffic = pd.read_csv("liva_hourly.csv")
weather = pd.read_csv("weather.csv")
kalio = pd.read_csv("kalio.csv")

In [11]:
# convert timestamps 
traffic["datetime"] = pd.to_datetime(traffic["datetime"]).dt.floor("h")
weather["datetime"] = pd.to_datetime(weather["datetime"]).dt.floor("h")

kalio["datetime"] = pd.to_datetime(
    kalio["Year"].astype(str) + "-" +
    kalio["Month"].astype(str) + "-" +
    kalio["Day"].astype(str) + " " +
    kalio["Time [Local time]"]
).dt.floor("h")
kalio["datetime"] = kalio["datetime"].dt.tz_localize("UTC")
kalio.columns

Index(['Observation station', 'Year', 'Month', 'Day', 'Time [Local time]',
       'Nitrogen dioxide [µg/m3]', 'Ozone [µg/m3]',
       'Inhalable particles <10 µm [µg/m3]', 'Fine particles <2.5 µm [µg/m3]',
       'Black carbon [µg/m3]', 'datetime'],
      dtype='object')

In [12]:
#find the traffic counter
import numpy as np

kais_lat = 60.1756
kais_lon = 24.9440

traffic["distance_to_kaisaniemi"] = np.sqrt(
    (traffic["lat"] - kais_lat)**2 +
    (traffic["lon"] - kais_lon)**2
)

closest_counter = (
    traffic.groupby("counterId")["distance_to_kaisaniemi"]
    .mean()
    .sort_values()
    .index[0]
)

closest_counter

np.int64(32)

In [13]:
traffic_single = traffic[traffic["counterId"] == closest_counter].copy()

In [17]:
# merge all dataset
merged = (
    traffic_single
    .merge(weather, on="datetime", how="inner")
    .merge(kalio, on="datetime", how="inner")
)
merged.isna().sum()

period                                    0
counterId                                 0
datetime                                  0
direction                             12696
unit                                      0
value                                     0
lat                                       0
lon                                       0
distance_to_kaisaniemi                    0
temperature                               6
pressure                                  6
humidity                                  6
precipitation                             9
snow_depth                              462
wind_speed                              322
cloud_cover                               5
Observation station                       0
Year                                      0
Month                                     0
Day                                       0
Time [Local time]                         0
Nitrogen dioxide [µg/m3]                  0
Ozone [µg/m3]                   

In [20]:
# Fill missing values with the average values
weather_cols = [
    "temperature",
    "pressure",
    "humidity",
    "precipitation",
    "snow_depth",
    "wind_speed",
    "cloud_cover"]

merged[weather_cols] = merged[weather_cols].interpolate(method="linear")

merged.isnull().sum()

period                                    0
counterId                                 0
datetime                                  0
direction                             12696
unit                                      0
value                                     0
lat                                       0
lon                                       0
distance_to_kaisaniemi                    0
temperature                               0
pressure                                  0
humidity                                  0
precipitation                             0
snow_depth                                0
wind_speed                                0
cloud_cover                               0
Observation station                       0
Year                                      0
Month                                     0
Day                                       0
Time [Local time]                         0
Nitrogen dioxide [µg/m3]                  0
Ozone [µg/m3]                   

In [21]:
merged.head()

,period,counterId,datetime,direction,unit,value,lat,lon,distance_to_kaisaniemi,temperature,...,Observation station,Year,Month,Day,Time [Local time],Nitrogen dioxide [µg/m3],Ozone [µg/m3],Inhalable particles <10 µm [µg/m3],Fine particles <2.5 µm [µg/m3],Black carbon [µg/m3]
0,hour,32,2023-11-15 12:00:00+00:00,NaN,n,21,60.17577,24.949638,0.005641,0.7,...,Helsinki Kallio 2,2023,11,15,12:00,12.4,37.8,17.4,6.3,0.2
1,hour,32,2023-11-15 13:00:00+00:00,NaN,n,19,60.17577,24.949638,0.005641,0.7,...,Helsinki Kallio 2,2023,11,15,13:00,17.7,33.9,17.6,5.8,0.2
2,hour,32,2023-11-15 14:00:00+00:00,NaN,n,21,60.17577,24.949638,0.005641,0.8,...,Helsinki Kallio 2,2023,11,15,14:00,17.5,34.3,18.1,5.6,0.3
3,hour,32,2023-11-15 15:00:00+00:00,NaN,n,27,60.17577,24.949638,0.005641,0.8,...,Helsinki Kallio 2,2023,11,15,15:00,18.6,34.9,18.8,6.5,0.3
4,hour,32,2023-11-15 16:00:00+00:00,NaN,n,20,60.17577,24.949638,0.005641,0.6,...,Helsinki Kallio 2,2023,11,15,16:00,21.3,33.5,20.4,5.5,0.4


In [22]:
cols_to_remove = [
    "period",
    "counterId",
    "direction",
    "unit",
    "distance_to_kaisaniemi",
    "Observation station",
    "Year",
    "Month",
    "Day",
    "Time [Local time]"
]

merged = merged.drop(columns=cols_to_remove)

In [24]:
merged.head()
merged.isnull().sum()

datetime                              0
value                                 0
lat                                   0
lon                                   0
temperature                           0
pressure                              0
humidity                              0
precipitation                         0
snow_depth                            0
wind_speed                            0
cloud_cover                           0
Nitrogen dioxide [µg/m3]              0
Ozone [µg/m3]                         0
Inhalable particles <10 µm [µg/m3]    0
Fine particles <2.5 µm [µg/m3]        0
Black carbon [µg/m3]                  0
dtype: int64

In [26]:
merged.to_csv("clean_dataset.csv", index=False)